# 📗 인증, 요청 한도, 에러 처리: 노무 상담 API를 vLLM 앞에서 보호하기

<div class="day77-callout" style="margin:10px 0;padding:8px 14px;font-size:17px;line-height:1.75;color:#202124;background:#ffffff;border-left:4px solid #54769b;border-radius:4px;"><strong style="display:block;font-size:1.13em;color:#1f3a5f;margin-bottom:2px;">실습 환경</strong><strong style="font-size:1em;color:#1f3a5f;">실행 위치:</strong> VS Code의 Remote - SSH로 접속한 RunPod 파드 하나에서 노트북, 터미널, 두 서버를 모두 실행합니다. <strong style="font-size:1em;color:#1f3a5f;">내 PC에서 실행하는 코드는 없습니다.</strong><br><strong style="font-size:1em;color:#1f3a5f;">준비:</strong> 교안 01의 본실습과 함께 따라하기를 마친 VS Code 원격 창에서 이어서 진행합니다.<br><strong style="font-size:1em;color:#1f3a5f;">서버:</strong> 터미널 1의 vLLM 서버는 <strong style="font-size:1em;color:#1f3a5f;">그대로 이어 씁니다.</strong> 터미널 2의 FastAPI 서버만 이 교안의 앱으로 바꿔 다시 띄웁니다.<br><strong style="font-size:1em;color:#1f3a5f;">최소 권장:</strong> GPU VRAM 24GB 1장(교안 01과 같은 파드).</div>

#### **API 보호 계층**

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요청이 vLLM에 닿기 전에 키, 한도, 양식을 확인하고, 실패를 정해진 양식으로 돌려주는 서빙 API의 앞단 코드</strong>입니다.</p>
<p style="margin:6px 0;">GPU 시간은 한정되어 있어서, 키가 없는 요청과 한 사용자가 몰아 보낸 요청은 GPU를 쓰기 전에 돌려보내야 합니다. 실패는 앱이 읽을 수 있는 양식으로 알려 줍니다.</p>
<p style="margin:6px 0;">여러 앱과 고객사에 키를 발급하는 사내 상담 API나, 요금제마다 사용량을 나누는 유료 모델 API가 이런 보호 계층을 둡니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/03_request_journey.png" width="1050" alt="요청 하나가 서빙 API에서 거치는 다섯 단계와, 앞 네 단계에서 실패할 때 돌려주는 상태 코드">

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 교안에서는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app.secure_main</code>을 실행합니다. 교안 01의 노무 상담 라우터는 그대로 두고, 라우터를 앱에 붙일 때 서비스 키 확인과 요청 한도 확인을 상담 함수 앞에 끼운 앱입니다(2절의 의존성). 401, 429, 422, 502, 504 실패는 모두 한 오류 양식으로 돌려주고, 환경 변수만 바꿔 vLLM이 늦거나 연결되지 않는 상황도 직접 만들어 봅니다. 과제 02에서는 같은 기술로 온라인 쇼핑 상담 API를 보호합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/20_lesson02_goals.png" width="1050" alt="학습 목표 그림: 1 서비스 키 인증, 2 요청 한도, 3 오류 응답 양식, 4 환경 변수 설정">

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">

<span class="day77-goal" style="font-size:1.13em;color:#1f3a5f;">**학습 목표**</span> (그림의 번호와 같습니다)

1. X-API-Key 헤더의 서비스 키를 의존성(Depends)으로 확인하고, 없거나 틀린 키를 401로 돌려보냅니다.
2. 요금제별 요청 한도를 넘긴 요청을 429와 Retry-After로 돌려보냅니다.
3. 오류 응답을 한 양식으로 맞추고, vLLM이 응답하지 않거나 늦을 때 502, 504로 바꿉니다.
4. 같은 코드를 환경 변수만 바꿔 다른 vLLM 주소와 시간 제한으로 실행합니다.

</div>

## 1. 실습 준비

### 1-1. 터미널 2에서 보호 계층 앱 실행

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1의 vLLM 서버는 그대로 둡니다.</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Application startup complete</code> 뒤로 오류 없이 로그가 이어지면 켜져 있는 것입니다. 파드를 새로 만들었거나 서버가 꺼졌다면 교안 01 1절(실습 준비와 vLLM 서버 실행)의 순서대로 키 생성과 vLLM 실행부터 마칩니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 2</strong>에서 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code> 키</strong>로 앞 노트북의 앱(과제 01의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api.main</code> 또는 교안 01의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app.main</code>)을 끄고, 같은 터미널에서 아래 두 블록으로 이 교안 폴더로 이동해 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app.secure_main</code>을 띄웁니다. uvicorn 명령은 교안 01과 같고 모듈 이름만 다릅니다.</p>
<p style="margin:6px 0;">이 교안 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app/</code>에는 교안 01의 완성 코드(함께 따라하기의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">routers/lease_router.py</code> 포함)와 이 교안의 파일 네 개가 들어 있습니다. 그래서 교안 01의 빈칸을 다 채우지 못했어도 이 교안을 진행할 수 있습니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">폴더, 파일</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">계층과 하는 일</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">보는 절</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dependencies/security.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">새 계층(의존성): 라우터보다 먼저 실행되어 서비스 키와 요청 한도를 확인합니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">2절, 3절</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">core/errors.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">설정과 공통 규칙: 실패를 한 오류 응답 양식으로 바꿉니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">4절, 5절</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">secure_main.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">앱 조립: 교안 01의 라우터에 의존성과 오류 양식을 붙인 앱입니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">1절, 2절</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">resources/service_keys.json</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">연습용 서비스 키와 요금제별 요청 한도입니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">2절, 3절</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">routers/</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">services/</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">schemas/</code>의 교안 01 파일은 그대로입니다. 인증과 요청 한도를 라우터 파일에 넣지 않고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dependencies/</code> 계층에 두어, 상담 함수는 고치지 않고 앱을 조립할 때 붙입니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 2</strong></div>

```bash
# 새 원격 터미널이면 이 블록부터 실행합니다
# 원격 저장소(실습자료) 맨 위 폴더를 찾아 그 안의 폴더로 이동합니다(저장소 안 어느 폴더에서 실행해도 됩니다)
cd "$(git rev-parse --show-toplevel)/day77_FastAPI_모델서빙API/교안_02_인증_요청한도_에러처리"
pwd   # 끝이 교안_02_인증_요청한도_에러처리이면 맞습니다
```

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 2</strong></div>

```bash
uv run uvicorn serving_app.secure_main:app --host 0.0.0.0 --port 8080 --reload
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">터미널 2에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Uvicorn running on http://0.0.0.0:8080</code>가 나오면 FastAPI 서버가 요청을 받을 수 있습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reload</code>를 붙였으므로 서버 코드(.py)를 저장하면 서버가 저절로 다시 시작합니다.</p>
</div>

### 1-2. 원격 노트북 준비

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">준비 셀은 교안 01의 준비 코드(경로, 서버 주소, JSONL 읽기, 노무 질문)에 세 가지를 더했습니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">vLLM 키와 주소: 5절에서 vLLM에 직접 요청할 때 씁니다.</li>
<li style="margin:3px 0;">서비스 키 두 개: 서버 패키지의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">resources/service_keys.json</code>에 든 연습용 키라 노트북에 그대로 적습니다. 실제 서비스에서는 사용자마다 따로 발급하고 공개하지 않습니다.</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">REQUEST_TIMEOUT</code>: 노트북이 FastAPI 응답을 기다리는 최대 시간입니다.</li>
</ul>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · RunPod 원격 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다</span></div>

In [ ]:
# ---- 라이브러리, 경로, 설정값(교안 01의 준비 셀에 vLLM 키와 주소를 더함) ----
import json
import os
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv

lesson_dir = Path(".").resolve()  # 이 노트북의 단원 폴더(정답 노트북은 "..")
day_dir = lesson_dir.parent                # data, images 가 있는 일차 폴더
data_dir = day_dir / "data"

# 터미널 1에서 생성해 원격 저장소의 .env에 저장한 vLLM 키를 읽습니다(vLLM에 직접 요청할 때 씁니다)
load_dotenv(day_dir.parent / ".env", override=True)
VLLM_API_KEY = os.environ["VLLM_API_KEY"]

SERVING_URL = "http://127.0.0.1:8080"   # 터미널 2에서 띄운 FastAPI 서빙 API
VLLM_URL = "http://127.0.0.1:8000/v1"  # 터미널 1에서 띄운 vLLM 서버(직접 요청할 때 씁니다)

# 테이블의 긴 글(질문, 답)이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# ---- JSONL 읽기(교안 01과 같은 함수) ----
def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(서빙 벤치마킹 단원과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


# ---- 노무 질문, 서비스 키, 요청 시간 제한 ----
labor_questions = load_jsonl(data_dir / "본실습" / "labor_questions.jsonl")

# 서버의 resources/service_keys.json 에 든 연습용 서비스 키(무료 요금제 60초에 3번, pro 요금제 60초에 30번)
FREE_KEY = "svc-free-demo-0001"
PRO_KEY = "svc-pro-demo-0002"

# 노트북이 FastAPI 응답을 기다리는 최대 시간(초). httpx 기본값 5초는 답 생성 시간보다 짧을 수 있습니다
REQUEST_TIMEOUT = 120

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">상태 확인 주소 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/healthz</code>는 키 없이 열어 둔 주소라, 서버가 요청을 받을 수 있는지 확인할 때 씁니다. 교안 01의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app.main</code>과 과제 01의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api.main</code>에도 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/healthz</code>가 있으므로, 이 셀만으로는 터미널 2에 어느 앱이 떠 있는지 알 수 없습니다.</p>
</div>

In [ ]:
# 키 없이 상태 확인 주소를 부릅니다(vLLM은 부르지 않습니다)
health_response = httpx.get(f"{SERVING_URL}/healthz", timeout=REQUEST_TIMEOUT)
print(f"상태 코드: {health_response.status_code} | 본문: {health_response.json()}")

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">상태 코드 200, 본문 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">{'status': 'ok'}</code>:</strong> 터미널 2의 서버가 요청을 받고 있습니다. 앞 노트북의 앱이 떠 있어도 같은 결과가 나옵니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">떠 있는 앱 구분:</strong> 내 PC 브라우저에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://localhost:8080/docs</code>를 열어 맨 위 제목이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">노무, 전월세 상담 API(서비스 키 필요)</code>인지 봅니다(교안 01 2-1에서 본 것처럼 앱의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">title</code>이 문서 화면 맨 위에 보입니다).</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">앞 앱이 남아 있다는 신호:</strong> 2-2에서 키 없음 행이 401이 아니라 200으로 오거나, 셀이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">KeyError: 'error'</code>로 멈추면 앞 노트북의 앱이 아직 8080 포트를 쓰고 있습니다. 터미널 2에서 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code> 키</strong>로 끄고 1-1의 두 블록을 다시 실행합니다.</li>
</ul>
</div>

## 2. 서비스 키 인증

### 2-1. X-API-Key 헤더와 의존성(Depends)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서빙 API가 요청을 보낸 사용자를 알아야 사용자마다 사용량을 따로 세고, 문제가 생긴 사용자 한 명만 막을 수 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서비스 키(service key)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">서빙 API가 사용자(앱, 고객사)마다 발급하는 문자열입니다. 앱은 서비스 키로 서빙 API를 부릅니다. vLLM 키(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>)는 FastAPI 서버가 vLLM을 부를 때 쓰는 다른 키라 앱에 나눠 주지 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">헤더(header):</strong> 요청 본문 밖에 붙는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">이름: 값</code> 한 줄입니다. 이 교안은 서비스 키를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">X-API-Key</code> 헤더에 담아 보냅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">401 Unauthorized:</strong> 키가 없거나 발급한 키가 아니라서 요청을 받지 않는다는 상태 코드입니다.</li>
</ul>
<p style="margin:6px 0;">발급한 키와 요금제는 서버 패키지의 키 파일 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">resources/service_keys.json</code>에 있습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">plans</code>는 요금제별 한도, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">keys</code>는 키마다의 사용자와 요금제입니다.</p>
</div>

```json
{
 "plans": {
  "free": {"requests_per_minute": 3},
  "pro": {"requests_per_minute": 30}
 },
 "keys": {
  "svc-free-demo-0001": {"user": "체험 사용자", "plan": "free"},
  "svc-pro-demo-0002": {"user": "상담 센터", "plan": "pro"}
 }
}
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">의존성(dependency)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">엔드포인트 함수보다 먼저 실행되어, 값을 넘기거나 요청을 거절하는 함수입니다.</li>
<li style="margin:3px 0;">키 확인 코드를 엔드포인트마다 쓸 필요가 없습니다. 라우터를 앱에 붙일 때 한 번만 붙이면 그 라우터의 모든 엔드포인트에 적용됩니다.</li>
</ul>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dependencies/security.py</code>는 서버가 뜰 때 키 파일을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SERVICE_KEYS</code> 딕셔너리로 한 번 읽어 두고, 요청마다 헤더의 키가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SERVICE_KEYS[&quot;keys&quot;]</code>에 있는지 확인합니다. 키를 확인하는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">require_service_key</code>와 요청 한도를 확인하는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">check_rate_limit</code>이 모두 이 파일에 있습니다(60초 동안의 요청 수를 세는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">check_rate_limit</code>의 몸통은 3-1에서 봅니다).</p>
</div>

```python
# dependencies/security.py
# APIKeyHeader: 요청에서 정한 이름(X-API-Key)의 헤더 값을 꺼낸다
# auto_error=False 면 헤더가 없을 때 오류 대신 None 이 들어와 401 을 직접 정할 수 있다
# APIKeyHeader 로 받으면 /docs 화면에 키를 넣는 Authorize 버튼이 생긴다
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)


async def require_service_key(api_key: str | None = Depends(api_key_header)):
    """X-API-Key 헤더의 서비스 키를 확인하고 {"key", "user", "plan"} 을 돌려준다. 없거나 틀리면 401."""
    if api_key is None or api_key not in SERVICE_KEYS["keys"]:
        raise HTTPException(status_code=401, detail="서비스 키가 없거나 올바르지 않습니다. X-API-Key 헤더를 확인하세요.")

    client = SERVICE_KEYS["keys"][api_key]
    return {"key": api_key, "user": client["user"], "plan": client["plan"]}


async def check_rate_limit(client: dict = Depends(require_service_key)):
    """키 확인을 먼저 거친 뒤, 최근 60초 요청 수가 요금제 한도 안이면 client 를 돌려준다. 넘으면 429."""
    limit = SERVICE_KEYS["plans"][client["plan"]]["requests_per_minute"]
    now = time.monotonic()

    # 1) 60초 창 안의 요청 시각만 남긴다
    #    기록에는 60초 안의 시각만 다시 저장하므로 키마다 많아야 한도만큼(free 3개, pro 30개)이다
    recent = []
    for sent_at in request_log.get(client["key"], []):
        if now - sent_at < WINDOW_SECONDS:
            recent.append(sent_at)

    # 2) 한도를 다 썼으면, 가장 오래된 요청이 창을 벗어날 때까지 남은 초(올림)를 Retry-After 로 알린다
    if len(recent) >= limit:
        retry_after = math.ceil(WINDOW_SECONDS - (now - recent[0]))
        raise HTTPException(
            status_code=429,
            detail=f"{client['plan']} 요금제는 {WINDOW_SECONDS}초에 {limit}번까지 요청할 수 있습니다.",
            headers={"Retry-After": str(retry_after)},
        )

    # 3) 통과한 요청만 기록한다(거절한 요청은 한도에 세지 않는다)
    recent.append(now)
    request_log[client["key"]] = recent
    return client
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">인자 자리에 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Depends(함수)</code>를 쓰면:</strong> FastAPI가 그 함수를 먼저 실행하고, 돌려받은 값을 인자에 넣습니다. 우리 코드는 그 함수를 직접 부르지 않습니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">check_rate_limit(client: dict = Depends(require_service_key))</code>: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">require_service_key</code>가 먼저 실행되고, 돌려준 딕셔너리가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">client</code>에 들어갑니다.</li>
<li style="margin:3px 0;">앞 함수가 401을 일으키면 거기서 멈추고, 뒤 함수는 실행되지 않습니다.</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">api_key = Depends(api_key_header)</code>도 같습니다. 이렇게 이어진 것이 <strong style="font-size:1.13em;color:#1f3a5f;">의존성 사슬</strong>입니다.</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/22_depends_chain.png" width="1050" alt="요청 하나가 api_key_header, require_service_key, check_rate_limit, labor_chat 순서로 실행되고, 앞 함수가 돌려준 값이 Depends로 다음 함수의 인자에 들어가며, 401이나 429가 나면 그 아래 함수와 vLLM은 실행되지 않는 흐름">

<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">dependencies=[Depends(함수)]</code>로 붙이면:</strong> 값은 넘기지 않고 실행만 합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">secure_main.py</code>는 라우터를 앱에 붙일 때 사슬의 마지막 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">check_rate_limit</code> 하나만 붙이고(아래 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">protected</code>), 그래도 키 확인, 요청 한도, 상담 함수 순서로 실행됩니다. 그래서 교안 01의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor_chat</code>은 고치지 않습니다.</p>
</div>

```python
# secure_main.py
"""교안 02 서빙 API(인증, 요청 한도, 오류 양식). 실행(교안 02 폴더): uv run uvicorn serving_app.secure_main:app --host 0.0.0.0 --port 8080 --reload

교안 01 의 라우터를 그대로 가져와, 붙일 때 의존성만 더한다. 교안 01 코드는 고치지 않는다.
"""
from contextlib import asynccontextmanager

from fastapi import Depends, FastAPI

from .core.errors import register_error_handlers
from .dependencies.security import check_rate_limit
from .routers.labor_router import labor_router
from .routers.lease_router import lease_router
from .routers.system_router import system_router
from .services.backend import vllm_client


@asynccontextmanager
async def lifespan(app: FastAPI):
    # vllm_client 는 backend.py 가 이미 만들었다(만들 때는 연결을 열지 않는다)
    try:
        yield  # 준비가 끝나면 FastAPI가 요청을 처리한다.
    finally:
        await vllm_client.close()  # 종료 때 요청 처리에 쓴 vLLM 연결풀을 닫는다.


app = FastAPI(
    title="노무, 전월세 상담 API(서비스 키 필요)",
    description="X-API-Key 헤더의 서비스 키를 확인하고, 요금제별 요청 한도를 지킨 요청만 vLLM 에 보냅니다.",
    version="2.0.0",
    lifespan=lifespan,
)

# 오류 응답 양식을 먼저 붙인다
register_error_handlers(app)

# 상태 확인 주소는 키 없이 열어 둔다(배포 뒤 자동 확인용)
app.include_router(system_router)

# 상담 주소에는 요청 한도 의존성을 붙인다. check_rate_limit 은 키 확인(require_service_key)을 먼저 거친다
protected = [Depends(check_rate_limit)]
app.include_router(labor_router, dependencies=protected)
app.include_router(lease_router, dependencies=protected)
```

### 2-2. 키 없이, 틀린 키로, 맞는 키로 보내기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">같은 노무 질문을 헤더만 바꿔 세 번 보냅니다. 무료 키는 3절에서 요청 한도를 확인할 때 쓰므로, 맞는 키 자리에는 pro 키를 씁니다.</p>
</div>

In [ ]:
import time

# 같은 노무 질문을 헤더 세 가지(키 없음, 틀린 키, pro 키)로 보냅니다
labor_question = labor_questions[0]["question"]
auth_cases = {
    "키 없음": {},
    "틀린 키": {"X-API-Key": "svc-wrong-key-9999"},   # 키 파일에 없는 키
    "pro 키": {"X-API-Key": PRO_KEY},
}

# 응답은 오류 양식을 비교할 때 다시 쓰려고 auth_responses 에 남깁니다
auth_responses = {}
auth_rows = []
for case, headers in auth_cases.items():
    start = time.perf_counter()
    response = httpx.post(f"{SERVING_URL}/v1/labor/chat", headers=headers, json={"question": labor_question}, timeout=REQUEST_TIMEOUT)
    elapsed = time.perf_counter() - start
    auth_responses[case] = response

    # 성공하면 모델의 답(테이블이 넓어지지 않게 앞 80자), 실패하면 error 항목의 code 와 message 를 본문 칸에 넣습니다
    body = response.json()
    if response.status_code == 200:
        summary = body["answer"][:80]
    else:
        summary = f"{body['error']['code']}: {body['error']['message']}"
    auth_rows.append({"경우": case, "상태 코드": response.status_code, "걸린 시간(초)": round(elapsed, 2), "본문": summary})

auth_table = pd.DataFrame(auth_rows)
display(auth_table)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">키 없음, 틀린 키 행의 401과 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">invalid_service_key</code>:</strong> 의존성에서 거절되어 vLLM을 부르지 않았습니다. 그래서 걸린 시간이 pro 키 행보다 훨씬 짧습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">pro 키 행의 200:</strong> 의존성을 통과해 교안 01과 같은 노무 답이 왔습니다.</li>
</ul>
</div>

### 2-3. /docs 화면에서 키 넣기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">APIKeyHeader</code>로 받은 헤더는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/docs</code> 문서 화면에도 인증 방식으로 등록됩니다. 내 PC 브라우저에서 문서 화면을 열어 키를 넣고 요청합니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day77-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://localhost:8080/docs</code> (VS Code 포트 전달, PORTS 탭에 8080이 있어야 합니다)</li>
<li style="margin:3px 0;"><strong class="day77-item" style="font-size:1em;color:#1f3a5f;">누를 곳:</strong> 오른쪽 위 <strong style="font-size:1.13em;color:#1f3a5f;">Authorize</strong> 버튼을 누르고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">X-API-Key</code> 칸에 pro 키(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">PRO_KEY</code> 값)를 붙여 넣은 뒤 <strong style="font-size:1.13em;color:#1f3a5f;">Authorize</strong>, <strong style="font-size:1.13em;color:#1f3a5f;">Close</strong>를 차례로 누릅니다.</li>
</ul>
<p style="margin:6px 0;">상담 주소 줄 끝의 자물쇠 표시는 키가 필요한 주소라는 뜻입니다. 상태 확인 묶음의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/healthz</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/models</code>에는 자물쇠가 없습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">secure_main.py</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">system_router</code>를 의존성 없이 붙였기 때문입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/models</code>는 vLLM에 모델 이름만 묻고 답을 만들지 않으므로, 교안 01 3-3처럼 서빙 API가 vLLM에 닿는지 확인할 때 씁니다. 키를 넣은 뒤 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">POST /v1/labor/chat</code>을 펼쳐 <strong style="font-size:1.13em;color:#1f3a5f;">Try it out</strong>으로 보내면 요청에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">X-API-Key</code> 헤더가 붙습니다.</p>
</div>

## 3. 요청 한도

### 3-1. 60초 창과 Retry-After

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">키 하나가 요청을 몰아 보내면 같은 GPU를 쓰는 다른 사용자의 지연이 길어집니다. 서빙 벤치마킹 단원에서 동시 요청 수가 늘면 TTFT(첫 토큰 지연)가 길어진 것과 같은 이유입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요청 한도(rate limit)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">키 하나가 정해진 시간 안에 보낼 수 있는 요청 수입니다. 이 서버는 키 파일의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">requests_per_minute</code>로 요금제마다 정합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">시간 창(window):</strong> 요청 수를 세는 구간입니다. 이 서버는 요청이 올 때마다 그 시각부터 60초 전까지 사이에 통과한 요청만 셉니다. 창이 시간을 따라 움직이므로, 통과한 요청은 60초가 지나면 창에서 빠집니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">429 Too Many Requests:</strong> 한도를 넘긴 요청을 받지 않는다는 상태 코드입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Retry-After:</strong> 429 응답에 붙는 헤더로, 몇 초 뒤에 다시 보내면 받을 수 있는지 알려 줍니다.</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/21_rate_limit_window.png" width="1050" alt="무료 키 요청 여섯 개의 시간축: 요청 시각마다 다시 잡히는 60초 창 안에 이미 세 개가 있는 30초와 61초 요청만 429와 Retry-After를 받습니다">

<p style="margin:6px 0;">한도가 60초에 3번인 무료 키로 0초, 10초, 20초, 30초, 60초, 61초에 요청을 보낸 경우를 따라가 봅니다.</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">0초, 10초, 20초 요청:</strong> 창 안에 통과한 요청이 0개, 1개, 2개라 한도 안입니다. 200으로 답하고 시각을 기록합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">30초 요청:</strong> 창 안에 이미 3개가 있어 429로 돌려보냅니다. 거절한 요청은 기록하지 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Retry-After:</strong> 30초입니다. 창 안에서 가장 오래된 0초 요청이 60초에 창을 벗어나기 때문입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">60초 요청:</strong> 0초 요청은 정확히 60초 전이라 창에서 빠집니다. 30초 요청은 기록하지 않았으므로, 창 안에는 10초, 20초 요청 2개만 남아 200으로 답합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">61초 요청:</strong> 61초에서 60초 전까지 사이에 10초, 20초, 60초 요청 3개가 있어 다시 429이고, Retry-After는 60 - (61 - 10) = 9초입니다. 1분마다 기록을 비우는 방식이었다면 61초 요청은 새 1분의 둘째 요청이라 통과했을 것입니다.</li>
</ol>
</div>

<div class="day77-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">Retry-After</strong> = 60 − (지금 − 창 안에서 가장 오래된 요청 시각)</div>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">소수는 올림(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">math.ceil</code>)합니다. 예를 들어 남은 시간이 29.4초면 30초로 알립니다. 내림해 29초로 알리면 그 시각에는 가장 오래된 요청이 아직 창 안이라 다시 429를 받습니다.</p>
<p style="margin:6px 0;">2-1에서 인자만 본 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">check_rate_limit</code>의 몸통이 이 계산을 합니다. 키마다 최근 요청 시각은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">request_log</code>에 모아 둡니다.</p>
</div>

```python
# dependencies/security.py
# 요청 한도를 세는 시간 창(초)
WINDOW_SECONDS = 60


# 키마다 최근 요청 시각(time.monotonic 값) 리스트. 서버 메모리에 두므로 서버를 다시 켜면 비워진다
request_log = {}


async def check_rate_limit(client: dict = Depends(require_service_key)):
    """키 확인을 먼저 거친 뒤, 최근 60초 요청 수가 요금제 한도 안이면 client 를 돌려준다. 넘으면 429."""
    limit = SERVICE_KEYS["plans"][client["plan"]]["requests_per_minute"]
    now = time.monotonic()

    # 1) 60초 창 안의 요청 시각만 남긴다
    #    기록에는 60초 안의 시각만 다시 저장하므로 키마다 많아야 한도만큼(free 3개, pro 30개)이다
    recent = []
    for sent_at in request_log.get(client["key"], []):
        if now - sent_at < WINDOW_SECONDS:
            recent.append(sent_at)

    # 2) 한도를 다 썼으면, 가장 오래된 요청이 창을 벗어날 때까지 남은 초(올림)를 Retry-After 로 알린다
    if len(recent) >= limit:
        retry_after = math.ceil(WINDOW_SECONDS - (now - recent[0]))
        raise HTTPException(
            status_code=429,
            detail=f"{client['plan']} 요금제는 {WINDOW_SECONDS}초에 {limit}번까지 요청할 수 있습니다.",
            headers={"Retry-After": str(retry_after)},
        )

    # 3) 통과한 요청만 기록한다(거절한 요청은 한도에 세지 않는다)
    recent.append(now)
    request_log[client["key"]] = recent
    return client
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">request_log</code>로 요청 수를 세는 방식은 다음과 같습니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">같은 키의 여러 창구:</strong> 기록은 키별로 남으므로, 같은 키라면 노무와 전월세 창구의 요청을 합쳐 셉니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">429로 거절한 요청:</strong> 한도에 세지 않습니다. 그래서 기다리지 않고 계속 보내도 다시 받을 수 있는 시각이 늦춰지지 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">422로 거절한 요청:</strong> 한도를 1번 씁니다. 의존성이 본문 양식 검사보다 먼저 실행되기 때문입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버 메모리의 기록:</strong> 서버를 다시 시작하면 기록이 비워집니다. 같은 앱을 서버 여러 대로 띄우면 서버마다 따로 세므로, 모든 서버가 함께 읽고 쓰는 저장소에 기록을 둬야 합니다.</li>
</ul>
</div>

### 3-2. 무료 키로 네 번 보내기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">무료 키로 노무 질문을 네 번 이어 보내고, 응답마다 상태 코드와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Retry-After</code> 헤더를 모읍니다. 앞 요청의 답을 받은 뒤 다음 요청을 보내므로, 보낸 시각은 답을 만드는 시간만큼 벌어집니다.</p>
</div>

In [ ]:
# 무료 키로 같은 질문을 네 번 이어 보냅니다(무료 요금제 한도는 60초에 3번)
limit_responses = []
limit_rows = []
first_sent = time.perf_counter()
for number in range(1, 5):
    # 첫 요청을 보낸 때부터 지난 초
    sent_at = time.perf_counter() - first_sent
    response = httpx.post(f"{SERVING_URL}/v1/labor/chat", headers={"X-API-Key": FREE_KEY}, json={"question": labor_questions[1]["question"]}, timeout=REQUEST_TIMEOUT)
    limit_responses.append(response)

    # Retry-After 헤더는 429 응답에만 있습니다(없으면 None)
    limit_rows.append({
        "순번": number,
        "보낸 시각(초)": round(sent_at, 1),
        "상태 코드": response.status_code,
        "Retry-After": response.headers.get("Retry-After"),
    })

limit_table = pd.DataFrame(limit_rows)
display(limit_table)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">순번 1~3의 200:</strong> 60초 창 안의 요청이 한도 3번 안입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">순번 4의 429와 Retry-After:</strong> 창 안에 이미 3번이 있습니다. 값은 실행마다 다르고, 첫 요청 뒤 흐른 시간이 짧을수록 큽니다.</li>
<li style="margin:3px 0;">이 셀을 60초 안에 다시 실행하면 앞 실행의 요청이 창에 남아 있어 첫 행부터 429가 나올 수 있습니다.</li>
</ul>
<p style="margin:6px 0;">넷째 요청의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Retry-After</code>를 위의 식으로 계산한 값과 비교합니다.</p>
</div>

In [ ]:
# 서버가 준 Retry-After 와, 60초에서 첫 요청 뒤 흐른 시간을 뺀 값을 나란히 봅니다
retry_after = int(limit_responses[3].headers["Retry-After"])
waited = limit_rows[3]["보낸 시각(초)"] - limit_rows[0]["보낸 시각(초)"]
print(f"Retry-After: {retry_after}초 | 60 - 첫 요청 뒤 흐른 시간({waited:.1f}초) = {60 - waited:.1f}초")

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">두 값:</strong> 계산한 값을 올림하면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Retry-After</code>와 거의 같습니다. 서버는 요청을 받은 시각으로 계산하므로 1초 안쪽으로 다를 수 있습니다.</li>
</ul>
</div>

### 3-3. Retry-After만큼 기다린 뒤 다시 보내기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">클라이언트는 429를 받으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Retry-After</code> 초만큼 기다린 뒤 다시 보냅니다. 기다리는 동안 셀이 멈춰 있는 것이 정상입니다.</p>
</div>

In [ ]:
# Retry-After 초만큼 기다린 뒤 같은 무료 키로 다시 보냅니다
time.sleep(retry_after)
retry_response = httpx.post(f"{SERVING_URL}/v1/labor/chat", headers={"X-API-Key": FREE_KEY}, json={"question": labor_questions[1]["question"]}, timeout=REQUEST_TIMEOUT)
print(f"{retry_after}초 기다린 뒤 상태 코드: {retry_response.status_code}")

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">200:</strong> 가장 오래된 첫 요청이 창을 벗어나 창 안의 요청이 다시 한도 3번보다 적어졌습니다.</li>
</ul>
</div>

## 4. 오류 응답 양식

### 4-1. 오류 처리 함수로 양식 맞추기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">오류를 받은 앱이 코드로 다음 행동을 정하려면, 상태 코드가 달라도 본문 모양이 같아야 합니다. FastAPI의 기본 오류 본문은 모양이 둘입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">HTTPException</code>으로 낸 401, 429는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;detail&quot;: &quot;문장&quot;}</code>이고, 교안 00 3절(요청 본문과 스키마)에서 본 422는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;detail&quot;: [{&quot;type&quot;: ..., &quot;loc&quot;: [...], &quot;msg&quot;: ...}]}</code> 목록입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">예외 처리 함수(exception handler)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">요청을 처리하다 난 예외(엔드포인트, 의존성, 요청 양식 검사)를 받아 응답으로 바꾸는 함수입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">app.add_exception_handler(예외 종류, 함수)</code>로 붙입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">오류 양식:</strong> 이 서버는 모든 오류를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;error&quot;: {&quot;code&quot;: ..., &quot;message&quot;: ...}}</code>로 돌려줍니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">code</code>는 프로그램이 읽는 짧은 이름, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">message</code>는 사람이 읽는 문장입니다.</li>
<li style="margin:3px 0;">예를 들어 키 없는 요청의 기본 본문 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;detail&quot;: &quot;서비스 키가 없거나 올바르지 않습니다. ...&quot;}</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;error&quot;: {&quot;code&quot;: &quot;invalid_service_key&quot;, &quot;message&quot;: &quot;서비스 키가 없거나 올바르지 않습니다. ...&quot;}}</code>로 바뀝니다.</li>
</ul>
</div>

```python
# core/errors.py
# 상태 코드마다 프로그램이 읽을 짧은 오류 이름
ERROR_CODES = {
    401: "invalid_service_key",
    403: "admin_only",  # 교안 03 의 관리자 주소에서 쓴다
    404: "not_found",
    429: "rate_limited",
}


def error_response(status_code, code, message, headers=None):
    """오류 양식의 JSON 응답을 돌려준다."""
    return JSONResponse(
        status_code=status_code,
        content={"error": {"code": code, "message": message}},
        headers=headers,
    )


async def handle_http_error(request, exc):
    """HTTPException(401, 404, 429 등)을 오류 양식으로 바꾼다. Retry-After 같은 헤더는 그대로 넘긴다."""
    code = ERROR_CODES.get(exc.status_code, "http_error")
    return error_response(exc.status_code, code, str(exc.detail), exc.headers)


async def handle_validation_error(request, exc):
    """요청 양식 오류(422)를 오류 양식으로 바꾼다. 문제가 여럿이어도 첫 번째 문제의 위치와 내용만 알려 준다."""
    first = exc.errors()[0]
    # loc 예: ("body", "question"). 마지막 값이 문제가 된 항목 이름이다
    field = first["loc"][-1]
    return error_response(422, "invalid_request", f"{field}: {first['msg']}")


def register_error_handlers(app: FastAPI):
    """앱에 오류 처리 함수를 붙인다. 예외 종류가 겹치면 더 좁은 종류(시간 초과)의 함수가 먼저 쓰인다."""
    # 없는 주소의 404 는 FastAPI 가 아니라 Starlette 가 일으킨다
    # FastAPI HTTPException 의 부모인 Starlette 쪽에 붙여 401, 429, 404 를 한 양식으로 맞춘다
    app.add_exception_handler(StarletteHTTPException, handle_http_error)
    app.add_exception_handler(RequestValidationError, handle_validation_error)
    # vLLM 쪽 실패(5절). APITimeoutError 는 APIConnectionError 의 한 종류지만 더 좁아서 먼저 쓰인다(시간 초과만 504)
    app.add_exception_handler(openai.APITimeoutError, handle_vllm_timeout)
    app.add_exception_handler(openai.APIConnectionError, handle_vllm_unreachable)
    app.add_exception_handler(openai.APIStatusError, handle_vllm_status_error)
```

### 4-2. 401, 429, 422 응답 비교

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">앞에서 받은 401, 429 응답과, 새로 보내는 짧은 질문 요청 두 개(pro 키, 키 없음)를 한 테이블로 모읍니다. 질문이 한 글자인 본문은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ChatRequest</code>의 최소 2자 조건을 어깁니다.</p>
</div>

In [ ]:
# 질문이 한 글자(최소 2자)인 본문을 pro 키로 한 번, 키 없이 한 번 보냅니다
short_body = {"question": "?"}
invalid_response = httpx.post(f"{SERVING_URL}/v1/labor/chat", headers={"X-API-Key": PRO_KEY}, json=short_body, timeout=REQUEST_TIMEOUT)
no_key_invalid_response = httpx.post(f"{SERVING_URL}/v1/labor/chat", json=short_body, timeout=REQUEST_TIMEOUT)

# 앞에서 받은 401, 429 응답과 함께 모읍니다
error_cases = {
    "키 없음": auth_responses["키 없음"],
    "틀린 키": auth_responses["틀린 키"],
    "한도 초과(무료 키 넷째)": limit_responses[3],
    "짧은 질문(pro 키)": invalid_response,
    "짧은 질문(키 없음)": no_key_invalid_response,
}

# 경우마다 상태 코드, error 항목의 두 값, Retry-After 헤더를 한 행으로 만듭니다
error_rows = []
for case, response in error_cases.items():
    error = response.json()["error"]
    error_rows.append({
        "경우": case,
        "상태 코드": response.status_code,
        "code": error["code"],
        "message": error["message"],
        "Retry-After": response.headers.get("Retry-After"),
    })

error_table = pd.DataFrame(error_rows)
display(error_table)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">code</code>, <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">message</code> 열:</strong> 상태 코드가 달라도 모든 행이 같은 두 항목으로 옵니다. 앱은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">error.code</code> 하나로 다음 행동(키 확인, 기다렸다 다시 보내기, 입력 고치기)을 고릅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">짧은 질문(키 없음) 행의 401:</strong> 본문도 틀렸지만 키 확인 의존성이 본문 검사보다 먼저라 422가 아니라 401입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">한도 초과 행에만 있는 Retry-After:</strong> 오류 처리 함수가 429의 헤더를 그대로 넘겼습니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 순서:</strong> 이 서버는 요청 하나를 아래 순서로 확인하고, 앞 단계에서 걸린 요청은 뒤 단계로 보내지 않습니다(맨 위 그림과 같은 순서입니다).</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">단계</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">확인하는 것</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">실패하면</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">1. 키 확인</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">X-API-Key</code> 헤더가 있고 발급한 키인가</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">401</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">2. 요청 한도</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이 키가 60초 안에 보낸 요청 수가 한도 안인가</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">429와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Retry-After</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">3. 양식 검사</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">요청 본문이 스키마에 맞는가</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">422</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">4. vLLM 호출</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">첫 조각 전에 vLLM에 연결되고, 정해진 시간 안에 답하는가</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">502(연결 실패), 504(시간 초과). 5절에서 확인합니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">5. 답변</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">-</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">200과 함께 답(스트리밍이면 조각)을 보냅니다</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">GPU를 쓰지 않는 요청:</strong> 1~3단계에서 돌려보낸 요청은 vLLM에 닿지 않아 GPU를 쓰지 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">JSON이 깨진 요청:</strong> 본문을 JSON으로 읽을 수 없으면 FastAPI는 키 확인보다 먼저 422를 돌려줍니다. 위 순서는 본문이 JSON으로 읽히는 요청의 순서입니다.</li>
</ul>
</div>

## 5. vLLM이 응답하지 않을 때

### 5-1. 502, 504와 환경 변수

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">키, 한도, 양식을 통과한 요청도 vLLM 쪽 문제로 실패할 수 있습니다. 교안 01 3-3의 결과 읽기에서 본 것처럼, 교안 01의 앱은 vLLM에 닿지 못하면 500(Internal Server Error)을 돌려줍니다. 500은 서버 안에서 처리하지 못한 오류가 났다는 뜻일 뿐입니다. 이 응답을 받은 앱은 서빙 API 코드의 버그인지 vLLM이 멈춘 것인지 구분할 수 없고, 다시 보내도 될지도 정할 수 없습니다. 그래서 이 교안의 앱은 vLLM 쪽 실패를 502와 504로 나눠 정해진 오류 양식으로 알립니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">502 Bad Gateway와 504 Gateway Timeout</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">502:</strong> 서빙 API가 뒤의 서버(vLLM)에 연결하지 못했거나, 뒤의 서버에서 오류 응답을 받았다는 상태 코드입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">504:</strong> 뒤의 서버가 정한 시간 안에 답하지 않았다는 상태 코드입니다.</li>
<li style="margin:3px 0;">이 서버는 vLLM에 연결하지 못했거나 vLLM이 오류로 답하면 502, 시간 제한(이 서버의 기본값 120초) 동안 vLLM에서 응답이 오지 않으면 504로 돌려줍니다.</li>
</ul>
</div>

```python
# core/errors.py
async def handle_vllm_timeout(request, exc):
    """vLLM 이 시간 제한 안에 답하지 않으면(openai.APITimeoutError) 504."""
    return error_response(504, "backend_timeout", "모델 서버가 제한 시간 안에 답하지 않았습니다. 잠시 뒤 다시 시도하세요.")


async def handle_vllm_unreachable(request, exc):
    """vLLM 서버에 연결하지 못하면(openai.APIConnectionError) 502."""
    return error_response(502, "backend_unavailable", "모델 서버에 연결할 수 없습니다. 잠시 뒤 다시 시도하세요.")


async def handle_vllm_status_error(request, exc):
    """vLLM 이 오류 상태 코드로 답하면(openai.APIStatusError) 502. vLLM 의 상태 코드만 알리고 내부 메시지는 넘기지 않는다."""
    return error_response(502, "backend_error", f"모델 서버가 오류({exc.status_code})로 답했습니다.")
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">환경 변수(environment variable)</strong>는 프로그램 밖에서 이름과 값으로 정해, 실행하는 프로세스에 넘기는 설정값입니다. 코드를 고치지 않고 값만 바꿔 같은 서버를 다르게 띄울 수 있습니다.</p>
<p style="margin:6px 0;">셸에서 명령 앞에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">이름=값</code>을 붙이면 그 명령으로 띄운 서버에만 값이 들어갑니다. 서버를 끄면 값도 사라지므로, 다음에 환경 변수 없이 띄우면 기본값으로 돌아갑니다(5-4).</p>
<p style="margin:6px 0;">서버는 뜰 때 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">core/config.py</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Settings</code>(교안 01 4-2)로 이 값을 읽습니다. 이름의 대소문자는 가리지 않아 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_TIMEOUT_SECONDS</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm_timeout_seconds</code> 필드를 채우고, 셸에서 준 값이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>와 기본값보다 먼저 쓰입니다.</p>
<p style="margin:6px 0;">아래 두 환경 변수로 vLLM이 늦는 상황과 연결되지 않는 상황을 만듭니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">services/backend.py</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm_client</code>가 이 두 값으로 만들어지고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_retries=0</code>이라 실패한 요청을 다시 보내지 않습니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">환경 변수(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Settings</code> 필드)</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">기본값</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">바꿔 줄 값</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">만드는 상황</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">상태 코드</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_TIMEOUT_SECONDS</code>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm_timeout_seconds</code>)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">120</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">1</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">생각을 켠 답이 1초 안에 끝나지 않습니다</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">504</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_BASE_URL</code>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm_base_url</code>)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://127.0.0.1:8000/v1</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://127.0.0.1:8999/v1</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">아무 서버도 듣지 않는 포트에 연결합니다</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">502</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">모델 배포 단원에서도 코드는 그대로 두고, 이 두 환경 변수에 배포한 vLLM의 주소와 시간 제한만 넣어 실행합니다.</p>
</div>

### 5-2. 시간 제한 1초로 실행하기(504)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 2</strong>에서 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code> 키</strong>로 서버를 끄고, 시간 제한만 1초로 바꿔 다시 띄웁니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 2</strong></div>

```bash
VLLM_TIMEOUT_SECONDS=1 uv run uvicorn serving_app.secure_main:app --host 0.0.0.0 --port 8080 --reload
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Application startup complete</code>가 나오면 생각을 켠(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">think=True</code>) 노무 질문을 pro 키로 보냅니다. 2-2의 pro 키 행처럼 생각을 끈 답도 1초보다 오래 걸리지만, 생각을 켜면 답 앞에 생각 토큰까지 만들므로 1초 제한을 확실히 넘깁니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · RunPod 원격 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다</span></div>

In [ ]:
# 생각을 켠 질문을 보내고 걸린 시간을 계산합니다(서버의 vLLM 시간 제한은 1초)
start = time.perf_counter()
timeout_response = httpx.post(f"{SERVING_URL}/v1/labor/chat", headers={"X-API-Key": PRO_KEY}, json={"question": labor_question, "think": True}, timeout=REQUEST_TIMEOUT)
elapsed = time.perf_counter() - start
print(f"상태 코드: {timeout_response.status_code} | 걸린 시간: {elapsed:.1f}초 | 본문: {timeout_response.json()}")

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">504와 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">backend_timeout</code>:</strong> 걸린 시간이 약 1초입니다. FastAPI가 vLLM의 답을 1초까지만 기다린 뒤 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">APITimeoutError</code>를 받았고, 오류 처리 함수가 504로 바꿨습니다.</li>
</ul>
</div>

### 5-3. 틀린 vLLM 주소로 실행하기(502)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">터미널 2에서 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code> 키</strong>로 서버를 끄고, 이번에는 vLLM 주소만 아무도 듣지 않는 8999 포트로 바꿔 띄웁니다. 터미널 1의 vLLM 서버는 끄지 않습니다.</p>
<p style="margin:6px 0;">이 명령에는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_TIMEOUT_SECONDS</code>를 주지 않았으므로 시간 제한은 기본값 120초로 돌아갑니다. 환경 변수는 그 명령으로 띄운 서버에만 들어가기 때문입니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 2</strong></div>

```bash
VLLM_BASE_URL=http://127.0.0.1:8999/v1 uv run uvicorn serving_app.secure_main:app --host 0.0.0.0 --port 8080 --reload
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Application startup complete</code>가 나오면 같은 질문을 일반 주소와 스트리밍 주소로 보내고, vLLM에도 직접 모델 목록을 물어 vLLM 서버가 켜져 있는지 확인합니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · RunPod 원격 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다</span></div>

In [ ]:
# 같은 질문을 FastAPI의 일반 주소와 스트리밍 주소로 보냅니다
backend_rows = []
for path in ["/v1/labor/chat", "/v1/labor/chat/stream"]:
    response = httpx.post(f"{SERVING_URL}{path}", headers={"X-API-Key": PRO_KEY}, json={"question": labor_question}, timeout=REQUEST_TIMEOUT)
    error = response.json()["error"]
    backend_rows.append({"요청한 곳": f"FastAPI {path}", "상태 코드": response.status_code, "본문": f"{error['code']}: {error['message']}"})

# vLLM에 직접 모델 목록을 묻습니다. vLLM은 키를 Authorization 헤더에 'Bearer 키' 모양으로 받습니다
# openai 라이브러리가 대신 붙여 주던 헤더이고, 키 값은 출력하지 않습니다
vllm_response = httpx.get(f"{VLLM_URL}/models", headers={"Authorization": f"Bearer {VLLM_API_KEY}"}, timeout=REQUEST_TIMEOUT)
backend_rows.append({"요청한 곳": "vLLM /v1/models", "상태 코드": vllm_response.status_code, "본문": vllm_response.json()["data"][0]["id"]})

backend_table = pd.DataFrame(backend_rows)
display(backend_table)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">FastAPI 두 주소의 502와 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">backend_unavailable</code>:</strong> 8999 포트에서 연결이 거부되어 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">APIConnectionError</code>가 났고, 오류 처리 함수가 502로 바꿨습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM /v1/models 행의 200:</strong> vLLM 서버는 켜져 있습니다. 502의 원인은 FastAPI에 준 주소입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">스트리밍 주소의 502:</strong> 교안 01 6-1의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor_chat_stream</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">StreamingResponse</code>를 만들기 전에 스트림을 엽니다. 그래서 연결이 안 되면 응답을 시작하기 전에 예외가 나고, 오류 처리 함수가 502로 바꿉니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">첫 조각 뒤의 실패:</strong> 교안 00 5-2에서 본 것처럼 상태 코드 200과 헤더가 이미 나갔으므로 상태 코드를 바꿀 수 없고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[DONE]</code> 줄 없이 연결이 끊깁니다. 클라이언트는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[DONE]</code>을 받았는지로 끝까지 받았는지 확인합니다.</li>
</ul>
</div>

### 5-4. 기본 설정으로 다시 실행하기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">터미널 2에서 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code> 키</strong>로 서버를 끄고, 환경 변수 없이 처음 명령으로 다시 띄웁니다. 틀린 주소로 띄운 서버를 그대로 두지 않습니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 2</strong></div>

```bash
uv run uvicorn serving_app.secure_main:app --host 0.0.0.0 --port 8080 --reload
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">터미널 2에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Uvicorn running on http://0.0.0.0:8080</code>가 나오면 FastAPI 서버가 요청을 받을 수 있습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reload</code>를 붙였으므로 서버 코드(.py)를 저장하면 서버가 저절로 다시 시작합니다.</p>
</div>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버는 끄지 않습니다.</strong> 과제 02에서 터미널 1의 vLLM 서버를 그대로 쓰고, 터미널 2만 과제 02 폴더로 이동해 과제 앱으로 바꿔 띄웁니다. 오늘 실습을 여기서 마칠 때만 아래 순서로 끕니다.</p>
</div>

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">오늘 실습을 여기서 마칠 때</summary>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 2</strong>(FastAPI 서버)와 <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1</strong>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">qwen3.5-4b</code> vLLM 서버)에서 차례로 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code> 키</strong>를 누릅니다. 명령을 입력할 수 있게 되면 터미널 1에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">nvidia-smi</code>를 실행합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">GPU 메모리 사용량(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Memory-Usage</code>)이 서버를 띄우기 전 수준으로 줄었으면</strong> vLLM 서버가 꺼진 것입니다.</p>
<p style="margin:6px 0;">오늘 실습을 마치면 RunPod 콘솔에서 파드를 종료(Terminate)합니다. 정지(Stop)해도 다음에 같은 GPU로 다시 켜지지 않는 경우가 많습니다. 남길 파일(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output</code> 등)이 있으면 종료 전에 내 PC로 받아 둡니다.</p>
</div>

</details>